# Task 2.3 — Build Baggage Journey and Misconnect Risk

For each bag: construct complete scan journey from CHECK_IN to CAROUSEL delivery.
- Compute: `time_from_checkin_to_aircraft_load_mins`, `has_scan_exception`
- For transfer bags (`connecting_flight_id IS NOT NULL`):
  - `connection_time_available = outbound_scheduled_departure − inbound_actual_arrival`
  - Join `airline_sla` for domestic/international threshold
  - Flag `AT_RISK_MISCONNECT` if `connection_time_available < threshold`


In [ ]:
# ── Imports & Configuration ────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

DATA_DIR = "/FileStore/airport_analytics_data"
BRONZE_DIR = "/FileStore/delta_lake/bronze"
SILVER_DIR = "/FileStore/delta_lake/silver"

try:
    spark
except NameError:
    spark = (SparkSession.builder
        .appName("Task_2_3_Baggage_Journey")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate())


## Step 1 — Read Bronze Tables

In [ ]:
# ── Read Bronze tables ────────────────────────────────────────────
baggage_df = spark.read.format("delta").load(f"{BRONZE_DIR}/baggage_scans")
flight_ops_df = spark.read.format("delta").load(f"{SILVER_DIR}/flight_operations")
sla_df = spark.read.format("delta").load(f"{BRONZE_DIR}/airline_sla")

print(f"Baggage scans: {baggage_df.count()}")
print(f"Flight operations: {flight_ops_df.count()}")
print(f"Airline SLA: {sla_df.count()}")


## Step 2 — Construct Bag Journey (Pivot Scan Locations)

In [ ]:
# ── Pivot scan locations to build bag journey ────────────────────
SCAN_LOCATIONS = [
    "CHECK_IN", "SCREENING", "MAKEUP_BELT", "AIRCRAFT_LOAD",
    "AIRCRAFT_UNLOAD", "CAROUSEL", "TRANSFER_BELT"
]

bag_journey_df = (baggage_df
    .groupBy("bag_tag_number", "passenger_id", "flight_id", "connecting_flight_id")
    .pivot("scan_location", SCAN_LOCATIONS)
    .agg(F.first("scan_ts"))
)

# Rename columns
for loc in SCAN_LOCATIONS:
    bag_journey_df = bag_journey_df.withColumnRenamed(loc, f"{loc.lower()}_ts")

# Compute time from check-in to aircraft load
bag_journey_df = bag_journey_df.withColumn(
    "time_checkin_to_load_mins",
    F.round(
        (F.unix_timestamp("aircraft_load_ts") -
         F.unix_timestamp("check_in_ts")) / 60, 1
    )
)

# Check for scan exceptions
exception_bags = (baggage_df
    .filter(F.col("scan_status").isin("EXCEPTION", "MISREAD"))
    .select("bag_tag_number")
    .distinct()
    .withColumn("has_scan_exception", F.lit(True))
)

bag_journey_df = bag_journey_df.join(exception_bags, on="bag_tag_number", how="left")
bag_journey_df = bag_journey_df.withColumn(
    "has_scan_exception",
    F.coalesce(F.col("has_scan_exception"), F.lit(False))
)

# Flag transfer bags
bag_journey_df = bag_journey_df.withColumn(
    "is_transfer",
    F.col("connecting_flight_id").isNotNull() & (F.col("connecting_flight_id") != "")
)

print(f"Bag journeys constructed: {bag_journey_df.count()}")
print(f"Transfer bags: {bag_journey_df.filter('is_transfer').count()}")
bag_journey_df.show(5, truncate=False)


## Step 3 — Compute Misconnect Risk for Transfer Bags

In [ ]:
# ── Misconnect Risk for transfer bags ─────────────────────────────
transfer_bags_df = bag_journey_df.filter("is_transfer = true")

# Get inbound flight arrival time
inbound_times = (flight_ops_df
    .select(
        F.col("flight_id").alias("inbound_fid"),
        F.col("scheduled_arrival").alias("inbound_arrival"),
        F.col("pushback_ts").alias("inbound_actual_arrival"),
        F.col("airline_code").alias("inbound_airline")
    )
)

# Get outbound flight departure time
outbound_times = (flight_ops_df
    .select(
        F.col("flight_id").alias("outbound_fid"),
        F.col("scheduled_departure").alias("outbound_departure"),
        F.col("destination_iata")
    )
)

# Join transfer bags with flight times
transfer_risk_df = (transfer_bags_df
    .join(inbound_times, transfer_bags_df.flight_id == inbound_times.inbound_fid, "left")
    .join(outbound_times, transfer_bags_df.connecting_flight_id == outbound_times.outbound_fid, "left")
)

# Compute connection time available
transfer_risk_df = transfer_risk_df.withColumn(
    "connection_time_available_mins",
    F.round(
        (F.unix_timestamp("outbound_departure") -
         F.unix_timestamp(F.coalesce("inbound_actual_arrival", "inbound_arrival"))
        ) / 60, 1
    )
)

# Join SLA for connection thresholds (use domestic threshold as default)
transfer_risk_df = transfer_risk_df.join(
    sla_df.select("airline_code", "connection_time_domestic_mins",
                  "connection_time_international_mins"),
    transfer_risk_df.inbound_airline == sla_df.airline_code,
    "left"
)

# Use domestic threshold as default
transfer_risk_df = transfer_risk_df.withColumn(
    "connection_threshold_mins",
    F.col("connection_time_domestic_mins")
)

# Flag AT_RISK_MISCONNECT
transfer_risk_df = transfer_risk_df.withColumn(
    "misconnect_risk",
    F.when(
        F.col("connection_time_available_mins") < F.col("connection_threshold_mins"),
        "AT_RISK_MISCONNECT"
    ).otherwise("OK")
)

at_risk_count = transfer_risk_df.filter("misconnect_risk = 'AT_RISK_MISCONNECT'").count()
print(f"Transfer bags analyzed: {transfer_risk_df.count()}")
print(f"AT_RISK_MISCONNECT bags: {at_risk_count}")


## Step 4 — Combine & Write to Silver Delta

In [ ]:
# ── Combine all bags and write ────────────────────────────────────
# Non-transfer bags
non_transfer_df = (bag_journey_df
    .filter("is_transfer = false")
    .withColumn("connection_time_available_mins", F.lit(None).cast("double"))
    .withColumn("connection_threshold_mins", F.lit(None).cast("double"))
    .withColumn("misconnect_risk", F.lit("N/A"))
)

# Transfer bags — select matching columns
transfer_final_df = (transfer_risk_df
    .select(
        "bag_tag_number", "passenger_id", "flight_id", "connecting_flight_id",
        "check_in_ts", "screening_ts", "makeup_belt_ts", "aircraft_load_ts",
        "aircraft_unload_ts", "carousel_ts", "transfer_belt_ts",
        "time_checkin_to_load_mins", "has_scan_exception", "is_transfer",
        "connection_time_available_mins", "connection_threshold_mins",
        "misconnect_risk"
    )
)

# Union
all_bags_df = non_transfer_df.select(transfer_final_df.columns).unionByName(transfer_final_df)
all_bags_df = all_bags_df.withColumn("processing_ts", F.current_timestamp())

silver_baggage_path = f"{SILVER_DIR}/baggage_journey"
(all_bags_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_baggage_path))

print(f"Written to {silver_baggage_path}")
result_df = spark.read.format("delta").load(silver_baggage_path)
print(f"Silver baggage_journey total: {result_df.count()}")
print("\nMisconnect Risk Distribution:")
result_df.groupBy("misconnect_risk").count().show()
